<p align="center"><a href="https://www.plus2net.com/python/dictionary-copy.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Python dict.copy(): Shallow Copies vs. deepcopy & References Practice

**Tutorial & Full Documentation:** [https://www.plus2net.com/python/dictionary-copy.php](https://www.plus2net.com/python/dictionary-copy.php)

Run cells in order in Google Colab. All dictionary copying mechanics, variable assignment vs shallow copies (`dict.copy()`) vs deep copies (`copy.deepcopy()`), the nested mutable sharing trap, staging modifications, and self-checking assertions are embedded directly in this notebook. Save a copy in Google Drive to keep your edits.

### 1. Assignment vs. Shallow Copying
Assignment (`=`) creates a shared alias, while `dict.copy()` creates an independent outer dictionary.

In [ ]:
original = {"name": "Alice", "score": 95}

# Assignment (=): shared alias
alias = original
alias["score"] = 90
print("After alias change, original score:", original["score"])  # 90

# Shallow Copy (copy()): independent top-level keys
shallow = original.copy()
shallow["score"] = 100
print("After shallow copy change, original score:", original["score"])  # 90 (unmodified!)
print("Shallow copy score:                      ", shallow["score"])   # 100


### 2. The Nested Sharing Trap in Shallow Copies
Shallow copies share inner mutable containers (like lists or sub-dictionaries).

In [ ]:
profile = {"user": "bob_dev", "roles": ["dev", "tester"]}
shallow_clone = profile.copy()

# Mutating the inner list through the copy affects the original
shallow_clone["roles"].append("admin")
print("Original roles modified by shallow copy:", profile["roles"])


### 3. Full Isolation with copy.deepcopy()
`copy.deepcopy()` recursively duplicates all nested objects, guaranteeing complete independence.

In [ ]:
import copy

master_data = {"user": "bob_dev", "roles": ["dev", "tester"], "meta": {"env": "staging"}}
deep_clone = copy.deepcopy(master_data)

deep_clone["roles"].append("admin")
deep_clone["meta"]["env"] = "production"

print("Master roles untouched:", master_data["roles"])  # ['dev', 'tester']
print("Master env untouched:  ", master_data["meta"]["env"])  # 'staging'


### 4. Real-World Recipe: Safe Function Parameter Overrides
Clone incoming parameter dictionaries before applying defaults to protect caller state.

In [ ]:
def connect_db(options: dict) -> dict:
    settings = options.copy()
    settings.setdefault("timeout", 30)
    settings.setdefault("retries", 3)
    return settings

user_opts = {"host": "db.local"}
configured = connect_db(user_opts)
print("Configured settings:", configured)
print("Caller user_opts:   ", user_opts)  # Intact


### 5. Self-Checking Verification Suite
Run unit assertions verifying top-level independence and deepcopy isolation.

In [ ]:
class CopyVerifier:
    @staticmethod
    def test_isolation(source: dict) -> bool:
        c = source.copy()
        return c is not source

v = CopyVerifier()
d = {"a": 1, "items": [10, 20]}
sc = d.copy()
dc = copy.deepcopy(d)

# Test 1: Outer independence
assert v.test_isolation(d) is True
sc["a"] = 99
assert d["a"] == 1

# Test 2: Shallow copy shares nested list
sc["items"].append(30)
assert 30 in d["items"]

# Test 3: Deepcopy isolates nested list
dc["items"].append(40)
assert 40 not in d["items"]

print("All dict.copy() unit assertions passed successfully! 100% verified.")
